In [ ]:
## Load the video and transform it into a series of images
import os
import cv2

vid = cv2.VideoCapture("./static/data/video/sample.mp4")

print("CV_CAP_PROP_FRAME_WIDTH: '{}'".format(vid.get(cv2.CAP_PROP_FRAME_WIDTH)))
print("CV_CAP_PROP_FRAME_HEIGHT : '{}'".format(vid.get(cv2.CAP_PROP_FRAME_HEIGHT)))
print("CAP_PROP_FPS : '{}'".format(vid.get(cv2.CAP_PROP_FPS)))
print("CAP_PROP_POS_MSEC : '{}'".format(vid.get(cv2.CAP_PROP_POS_MSEC)))
print("CAP_PROP_FRAME_COUNT  : '{}'".format(vid.get(cv2.CAP_PROP_FRAME_COUNT)))
print("CAP_PROP_BRIGHTNESS : '{}'".format(vid.get(cv2.CAP_PROP_BRIGHTNESS)))
print("CAP_PROP_CONTRAST : '{}'".format(vid.get(cv2.CAP_PROP_CONTRAST)))
print("CAP_PROP_SATURATION : '{}'".format(vid.get(cv2.CAP_PROP_SATURATION)))
print("CAP_PROP_HUE : '{}'".format(vid.get(cv2.CAP_PROP_HUE)))
print("CAP_PROP_GAIN  : '{}'".format(vid.get(cv2.CAP_PROP_GAIN)))
print("CAP_PROP_CONVERT_RGB : '{}'".format(vid.get(cv2.CAP_PROP_CONVERT_RGB)))

## Total frames = 600, duration = 60s => we can expect 10 frames per second (for example, when reading the video stream)
# Read all frames (24 frames per second?) or how many frames per second?
frames_folder = "./static/data/frames/"
count, success = 0, True
while success:
    success, image = vid.read() # Read frame
    if success:
        cv2.imwrite(os.path.join(frames_folder, f"frame_{count}.jpg"), image)
        count += 1
vid.release()

Video notes:
- moving camera
- moving objects
- players colours (two teams), goalkeepers (different colours), referees
    - different lighting and shades

Latency requirements notes:
- 10 frames per second => 100ms per frame... ouch, that's quick, or is it?
- we need something that would speed up processing?

Approach 1:
 1. detect background and foreground
 2. detect objects on the foreground, i.e., field
 3. attribute each object to a team based on colours (configured before)

Approach 2:
 1. detect objects (humans) right away:
    - first, extract features as bounding boxes for humans on the screen (classical cv, feature detection)
    - then, classifier fine-tuned to detect players on the field

In [ ]:
## Experimenting: detecting movement -> pointless because camera moves
import matplotlib.pyplot as plt

def detect_motion(frame1, frame2):
    # Calculate difference between frames
    diff = cv2.absdiff(frame1, frame2)

    # Convert to grayscale
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

    # Apply threshold
    _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)

    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Draw rectangles around motion areas
    for contour in contours:
        if cv2.contourArea(contour) > 500:
            (x, y, w, h) = cv2.boundingRect(contour)
            cv2.rectangle(frame1, (x, y), (x+w, y+h), (0, 255, 0), 2)

    return frame1

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, ax in enumerate(axes.flat):
    f1 = cv2.imread(os.path.join(frames_folder, f"frame_{i}.jpg"))
    f2 = cv2.imread(os.path.join(frames_folder, f"frame_{i+1}.jpg"))
    result = detect_motion(f1, f2)
    ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frames {i} → {i+1}')
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
## Experimenting: Detecting background
backSub = cv2.createBackgroundSubtractorMOG2()

# Warm up: feed frames to build the background model
cap = cv2.VideoCapture("./static/data/video/sample.mp4")
for i in range(100):  # first 100 frames = ~10 seconds at 10fps
    ret, frame = cap.read()
    if not ret:
        break
    backSub.apply(frame)  # discard output, just training

# Now apply to a frame you care about
ret, frame = cap.read()
fgMask = backSub.apply(frame)

plt.imshow(fgMask, cmap='gray')
plt.title('FG Mask after warm-up')
plt.axis('off')
plt.show()

In [ ]:
## Detecting bounding boxes for humans
import cv2
import imutils

# Initialize HOG descriptor with pre-trained person detector
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# Read the image and resize for better detection efficiency
image = cv2.imread(os.path.join(frames_folder, f"frame_{1}.jpg"))
image = imutils.resize(image, width=min(1920, image.shape[1]))

# Detect humans in the image
(humans, _) = hog.detectMultiScale(image, winStride=(8, 8), padding=(16, 16), scale=1.05)

# Draw rectangles around detected humans
for (x, y, w, h) in humans:
    cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)

print(f"Detected {len(humans)} humans")
# Save the result
plt.imshow(image)
plt.title('Humans')
plt.axis('off')
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

def on_the_field_filter(image, pt1, pt2, margin=15, threshold=0.4):
    """
    Detect whether a player is on the playing field by checking
    if the surrounding pixels (below, left, right of the bounding box) are green.
    """
    x1, y1 = pt1
    x2, y2 = pt2
    h, w = image.shape[:2]

    regions = []
    if y2 + margin < h:
        regions.append(image[y2:y2 + margin, max(0, x1):min(w, x2)])
    if x1 - margin >= 0:
        regions.append(image[max(0, y1):min(h, y2), x1 - margin:x1])
    if x2 + margin < w:
        regions.append(image[max(0, y1):min(h, y2), x2:x2 + margin])

    if not regions:
        return False

    pixels = np.vstack([r.reshape(-1, 3) for r in regions if r.size > 0])
    pixels_hsv = cv2.cvtColor(pixels.reshape(-1, 1, 3), cv2.COLOR_BGR2HSV).reshape(-1, 3)

    green_mask = (
            (pixels_hsv[:, 0] >= 36) & (pixels_hsv[:, 0] <= 86) &
            (pixels_hsv[:, 1] > 40) &
            (pixels_hsv[:, 2] > 40)
    )
    return green_mask.mean() > threshold


def referee_filter(image, pt1, pt2, threshold=0.35):
    """
    Detect whether a human is a referee by checking for black shorts.
    Samples the middle third of the bounding box (shorts area), strips green pixels,
    then checks if enough remaining pixels are dark (black = low HSV brightness).
    """
    x1, y1 = pt1
    x2, y2 = pt2
    h, w = image.shape[:2]

    box_h = y2 - y1
    if box_h < 80:
        return False  # too small to classify reliably

    # Middle third = shorts area
    shorts_y1 = y1 + box_h // 3
    shorts_y2 = y1 + 2 * box_h // 3
    region = image[max(0, shorts_y1):min(h, shorts_y2), max(0, x1):min(w, x2)]
    if region.size == 0:
        return False

    pixels_hsv = cv2.cvtColor(region, cv2.COLOR_BGR2HSV).reshape(-1, 3)

    # Remove green pixels (pitch bleed-in)
    green_mask = (
            (pixels_hsv[:, 0] >= 36) & (pixels_hsv[:, 0] <= 86) &
            (pixels_hsv[:, 1] > 40) &
            (pixels_hsv[:, 2] > 40)
    )
    non_green = pixels_hsv[~green_mask]
    if len(non_green) == 0:
        return False

    # Black = low brightness (V < 80), regardless of hue
    black_mask = non_green[:, 2] < 80
    return black_mask.mean() > threshold


image = cv2.imread(os.path.join(frames_folder, "frame_1.jpg"))
# image = cv2.imread(os.path.join(frames_folder, "frame_329.jpg"))

model = YOLO('yolov8n.pt')
results = model(image)

on_field, off_field, referees = 0, 0, 0
for box in results[0].boxes:
    if int(box.cls) == 0:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        if not on_the_field_filter(image, (x1, y1), (x2, y2)):
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)  # red = off field
            off_field += 1
        elif referee_filter(image, (x1, y1), (x2, y2)):
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 255), 2)  # yellow = referee
            referees += 1
        else:
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)   # green = player
            on_field += 1

print(f"Players: {on_field}, Referees: {referees}, Off field: {off_field}")

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('green=player, yellow=referee, red=off field')
plt.axis('off')
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO


def _boxes_overlap(a, b):
    """Returns True if two boxes (x1,y1,x2,y2) have any intersection."""
    return a[0] < b[2] and a[2] > b[0] and a[1] < b[3] and a[3] > b[1]


def on_the_field_filter(image, pt1, pt2, all_boxes, margin=15, threshold=0.4):
    """
    Stage 1: check if surrounding pixels are green.
    Stage 2: if stage 1 fails, still count as on-field if this box overlaps
             with any other box that passed stage 1 (player occluded by another).
    """
    x1, y1 = pt1
    x2, y2 = pt2
    h, w = image.shape[:2]

    regions = []
    if y2 + margin < h:
        regions.append(image[y2:y2 + margin, max(0, x1):min(w, x2)])
    if x1 - margin >= 0:
        regions.append(image[max(0, y1):min(h, y2), x1 - margin:x1])
    if x2 + margin < w:
        regions.append(image[max(0, y1):min(h, y2), x2:x2 + margin])

    if regions:
        pixels = np.vstack([r.reshape(-1, 3) for r in regions if r.size > 0])
        pixels_hsv = cv2.cvtColor(pixels.reshape(-1, 1, 3), cv2.COLOR_BGR2HSV).reshape(-1, 3)
        green_mask = (
            (pixels_hsv[:, 0] >= 36) & (pixels_hsv[:, 0] <= 86) &
            (pixels_hsv[:, 1] > 40) &
            (pixels_hsv[:, 2] > 40)
        )
        if green_mask.mean() > threshold:
            return True

    this_box = (x1, y1, x2, y2)
    return any(_boxes_overlap(this_box, other) for other in all_boxes)


def classify_person(image, mask_bin):
    """
    Classify an on-field person in two steps:

    Step 1 — preprocessing: remove dark-green grass pixels (HSV hue 36-86, S>40, V>40).

    Step 2 — pixel-fraction classification on remaining pixels:
      - yellow   (hue 20-35, S>80, V>80)                            -> 'referee'
      - lime/neon-green (hue 36-86, S>150, V>150) or blue (hue 87-130, S>60, V>60) -> 'goalkeeper'
      - red/dark-red (hue 0-15 or 155-180, S>50, V>30)              -> 'player_team_1'
      - white/grey (S<60, V>120)                                     -> 'player_team_2'
      - else                                                         -> 'unknown'
    """
    pixels = image[mask_bin == 1]
    if len(pixels) == 0:
        return 'unknown'

    pixels_hsv = cv2.cvtColor(pixels.reshape(-1, 1, 3), cv2.COLOR_BGR2HSV).reshape(-1, 3)
    p = pixels_hsv

    # # Step 1: remove dark-green grass pixels
    # grass_mask = (
    #     (pixels_hsv[:, 0] >= 36) & (pixels_hsv[:, 0] <= 86) &
    #     (pixels_hsv[:, 1] > 40) &
    #     (pixels_hsv[:, 2] > 40)
    # )
    # p = pixels_hsv[~grass_mask]
    # if len(p) == 0:
    #     return 'unknown'

    h, s, v = p[:, 0], p[:, 1], p[:, 2]

    # Step 2: classify by pixel fraction (threshold = 25%)

    # Yellow → referee
    is_yellow = (h >= 20) & (h <= 35) & (s > 80) & (v > 80)
    if is_yellow.mean() > 0.25:
        return 'referee'

    # Goalkeeper: vibrant lime/neon yellow-green (hi-vis) OR blue (dark or light)
    is_neon_green = (h >= 36) & (h <= 86) & (s > 150) & (v > 150)
    is_blue = (h >= 87) & (h <= 130) & (s > 60) & (v > 60)
    if (is_neon_green | is_blue).mean() > 0.25:
        return 'goalkeeper'

    # Red + dark red (wraps around 0/180): broad S/V to catch maroon and dark red
    is_red = ((h <= 15) | (h >= 155)) & (s > 50) & (v > 30)
    if is_red.mean() > 0.25:
        return 'player_team_1'

    # White + grey: low saturation, moderate-to-high brightness
    is_white_grey = (s < 60) & (v > 120)
    if is_white_grey.mean() > 0.25:
        return 'player_team_2'

    return 'unknown'


# image = cv2.imread(os.path.join(frames_folder, "frame_329.jpg"))
# image = cv2.imread(os.path.join(frames_folder, "frame_58.jpg"))

image = cv2.imread(os.path.join(frames_folder, "frame_443.jpg"))

h_img, w_img = image.shape[:2]

model = YOLO('yolov8n-seg.pt')
results = model(image)

overlay = image.copy()
segmented_crops = []

COLOURS = {
    'player_team_1': (0, 0, 200),
    'player_team_2': (200, 200, 200),
    'referee':       (0, 255, 255),
    'goalkeeper':    (255, 165, 0),
    'unknown':       (128, 0, 128),
    'off_field':     (0, 0, 255),
}

result = results[0]
masks = result.masks

# First pass: determine which boxes pass stage 1 (green surroundings)
stage1_on_field = set()
for i, box in enumerate(result.boxes):
    if int(box.cls) != 0:
        continue
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    h, w = image.shape[:2]
    regions = []
    margin = 15
    if y2 + margin < h:
        regions.append(image[y2:y2 + margin, max(0, x1):min(w, x2)])
    if x1 - margin >= 0:
        regions.append(image[max(0, y1):min(h, y2), x1 - margin:x1])
    if x2 + margin < w:
        regions.append(image[max(0, y1):min(h, y2), x2:x2 + margin])
    if regions:
        pixels = np.vstack([r.reshape(-1, 3) for r in regions if r.size > 0])
        pixels_hsv = cv2.cvtColor(pixels.reshape(-1, 1, 3), cv2.COLOR_BGR2HSV).reshape(-1, 3)
        green_mask = (
            (pixels_hsv[:, 0] >= 36) & (pixels_hsv[:, 0] <= 86) &
            (pixels_hsv[:, 1] > 40) & (pixels_hsv[:, 2] > 40)
        )
        if green_mask.mean() > 0.4:
            stage1_on_field.add(i)

confirmed_boxes = [
    tuple(map(int, result.boxes[i].xyxy[0])) for i in stage1_on_field
]

counts = {k: 0 for k in COLOURS}

for i, box in enumerate(result.boxes):
    if int(box.cls) != 0:
        continue

    x1, y1, x2, y2 = map(int, box.xyxy[0])

    mask_bin = None
    if masks is not None and i < len(masks):
        mask_bin = (masks[i].data[0].cpu().numpy() > 0.5).astype(np.uint8)
        mask_bin = cv2.resize(mask_bin, (w_img, h_img), interpolation=cv2.INTER_NEAREST)

    this_box = (x1, y1, x2, y2)
    is_on_field = (
        i in stage1_on_field or
        any(_boxes_overlap(this_box, other) for other in confirmed_boxes if other != this_box)
    )

    if not is_on_field:
        label = 'off_field'
    elif mask_bin is not None:
        label = classify_person(image, mask_bin)
    else:
        label = 'unknown'

    counts[label] += 1
    colour = COLOURS[label]

    cv2.rectangle(overlay, (x1, y1), (x2, y2), colour, 2)
    cv2.putText(overlay, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, colour, 1)

    if mask_bin is not None:
        coloured = np.zeros_like(image)
        coloured[:] = colour
        alpha = 0.4
        overlay[mask_bin == 1] = (
            alpha * coloured[mask_bin == 1] + (1 - alpha) * overlay[mask_bin == 1]
        ).astype(np.uint8)

        crop = image[max(0, y1):min(h_img, y2), max(0, x1):min(w_img, x2)].copy()
        mask_crop = mask_bin[max(0, y1):min(h_img, y2), max(0, x1):min(w_img, x2)]
        crop[mask_crop == 0] = 0
        conf = float(box.conf[0])
        segmented_crops.append({'label': label, 'crop': crop, 'mask': mask_crop, 'conf': conf})

print(counts)
print(f"Saved {len(segmented_crops)} segmented crops")

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title('red=team1, grey=team2, orange=goalkeeper, yellow=referee, purple=unknown, blue=off_field')
plt.axis('off')
plt.show()

if segmented_crops:
    n = len(segmented_crops)
    cols = min(n, 6)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 3))
    axes = np.array(axes).flatten()
    for j, item in enumerate(segmented_crops):
        axes[j].imshow(cv2.cvtColor(item['crop'], cv2.COLOR_BGR2RGB))
        axes[j].set_title(f"{item['label']}\n{item['conf']:.2f}", fontsize=7)
        axes[j].axis('off')
    for j in range(len(segmented_crops), len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()
